# ARK-019 V4 — Science-preserving, acquisition-qualified Capability Guardian

Select **T4 GPU**, then **Runtime → Run all**. V4 repairs the three blockers exposed by V3.1: an underpowered SKILL_B channel, science-damaged SKILL_A parents, and a 100-update controller observation delay. The fixed campaign may require multiple T4 sessions; rerun this same notebook until the final result ZIP is produced.


In [ ]:
# CELL 0 — live readiness -> frozen executable -> static tests -> Drive substrate gate
import json, subprocess, sys
from pathlib import Path
REPO=Path('/content/An-Ra-the-new-AGI-r3v4')
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='Arkenstone'
EXEC='8e858c614d764335100dfd41dda6f8e0d0c877a7'
READY_REL='experiments/ARK-019/RUN_READINESS_V4.json'
if not REPO.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','160',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH,'--depth','160'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'],check=True)
ready=json.loads((REPO/READY_REL).read_text())
assert ready['status']=='READY_FOR_OPERATOR_COLAB_CUDA_PREFLIGHT'
assert ready['scientific_result_status']=='NOT_EXECUTED'
assert ready['frozen_scientific_executable_commit']==EXEC
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXEC],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(); assert head==EXEC
expected={
 'experiments/ARK-019/PLAN_V4.md':'a63ede220585fbd90004fd2aff4c6c31b37ea0ce',
 'experiments/ARK-019/PREREGISTRATION_V4.json':'02dd7b3d08931fa31b97ed681164b14011a18bd4',
 'experiments/ARK-019/ark019_v4_core.py':'18133fd31e16de20e633ca86f5b88cb817b97371',
 'experiments/ARK-019/run_ark019_v4.py':'4adb1558e92270f596df8bccdb0b11478b9097d8',
 'tests/test_ark019_v4.py':'4a972076f187035dfaae1353e3d77c65443afc48',
 'experiments/ARK-019/run_ark019_v3.py':'ef41096069714668b3b1e5c0e165904908ebc885',
 'experiments/ARK-018/ark018_v3_common.py':'05b1c6a7832749740420b5b540c3214ef4c84492',
 'experiments/ARK-018/ark018_v3_binding_fast.py':'7492c697eccc5d87528094acf1b2e6164e47b1e1'}
for p,sha in expected.items():
    got=subprocess.check_output(['git','-C',str(REPO),'hash-object',p],text=True).strip(); assert got==sha,(p,got,sha)
pre=json.loads((REPO/'experiments/ARK-019/PREREGISTRATION_V4.json').read_text())
assert pre['status']=='PREREGISTERED_BEFORE_V4_SCIENTIFIC_EXECUTION'
assert pre['prior_evidence']['ark019_v3_bundle_sha256']=='fcc14c5378318b8c447c2735768b72920334d8c9292b445372fd17d3d2b554d8'
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers==0.21.4','pytest','numpy'],check=True)
for p in ['experiments/ARK-019/ark019_v4_core.py','experiments/ARK-019/run_ark019_v4.py','experiments/ARK-019/run_ark019_v3.py','experiments/ARK-018/ark018_v3_common.py','experiments/ARK-018/ark018_v3_binding_fast.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(REPO/p)],check=True)
subprocess.run([sys.executable,'-m','pytest',str(REPO/'tests/test_ark019_v4.py'),'-q'],cwd=REPO,check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1')
required=[ROOT/'prepared/ARK-018_PREPARED_RECEIPT.json',ROOT/'prepared/tokenizer.json',ROOT/'prepared/train.bin',ROOT/'prepared/control.bin',ROOT/'prepared/sealed.bin',ROOT/'prepared/token_counts.npy',ROOT/'checkpoints/seed_31801/SCIENCE_ONLY.pt',ROOT/'checkpoints/seed_31902/SCIENCE_ONLY.pt']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing ARK-018 prerequisite(s): '+repr(missing))
print('ARK-019 V4 STATIC + SUBSTRATE PREFLIGHT: PASS')
print('Frozen science:',EXEC,'| V4 may span multiple T4 sessions; protocol will not be reduced to fit one session.')


In [ ]:
# CELL 1 — run or resume the fixed V4 campaign
import subprocess, sys
cmd=[sys.executable,'experiments/ARK-019/run_ark019_v4.py','--mode','all']
print('Starting:', ' '.join(cmd), flush=True)
subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
# CELL 2 — report/download final result or exact-resumable partial session
import hashlib, json
from pathlib import Path
from google.colab import files
OUT=Path('/content/drive/MyDrive/genisis-arkenstone/ARK019_GUARDIAN_V4')
final=OUT/'ARKENSTONE_ARK019_V4_GUARDIAN_RESULTS.zip'
partial=OUT/'ARKENSTONE_ARK019_V4_GUARDIAN_PARTIAL.zip'
failure=OUT/'ARK-019_V4_FAILURE.json'
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(8<<20),b''): h.update(b)
    return h.hexdigest()
if final.exists():
    actual=sha(final); side=OUT/(final.name+'.sha256')
    if side.exists(): assert side.read_text().split()[0]==actual
    rp=OUT/'ARK-019_V4_RESULT.json'; r=json.loads(rp.read_text()) if rp.exists() else {}
    print('STATUS:',r.get('status')); print('VERDICT:',r.get('decision',{}).get('verdict',r.get('verdict'))); print('SELECTED B SLOTS:',r.get('selected_b_slots')); print('ZIP SHA256:',actual)
    if r.get('decision'): print(json.dumps(r['decision'],indent=2))
    files.download(str(final))
elif partial.exists():
    actual=sha(partial); side=OUT/(partial.name+'.sha256')
    if side.exists(): assert side.read_text().split()[0]==actual
    sp=OUT/'SESSION_STATE_V4.json'; state=json.loads(sp.read_text()) if sp.exists() else {}
    print('STATUS: PARTIAL_SESSION'); print(json.dumps(state,indent=2)); print('ZIP SHA256:',actual)
    print('Rerun THIS SAME notebook with T4 GPU. Existing Drive checkpoints will resume the frozen campaign.')
    files.download(str(partial))
else:
    print('No V4 bundle found.')
    if failure.exists(): print(failure.read_text())
    raise FileNotFoundError('Neither final nor partial V4 bundle exists; inspect the failure output above.')
